# Lab 07: Operação Churn Zero (CrewAI + Gemini 2.5)
**Disciplina:** Augmented Analytics & AI-Driven Insights  
**Objetivo:** Orquestrar uma Squad de IAs especialistas para reverter cancelamentos de clientes em tempo real.

---

### 🚨 CENÁRIO DE GUERRA
A diretoria detectou uma anomalia na retenção de clientes Premium. Você não tem tempo para analisar um por um. Sua missão é ativar uma **Squad de Agentes de IA** que irá:
1. Identificar o cliente de maior risco.
2. Criar uma oferta de retenção personalizada.
3. Escrever um e-mail executivo de alto impacto.

### BLOCO 1: Preparação do Quartel-General (Instalação)
*Neste bloco, instalamos o motor de orquestração (CrewAI) e as ferramentas do Google.*

In [ ]:
# Instalação das bibliotecas (Pode levar de 2 a 3 min)
!pip install -q crewai google-generativeai langchain-google-genai
print("✅ Ambiente configurado com sucesso!")

### BLOCO 2: Conexão com o Comando Central (API Key)
*Aqui configuramos a chave e o nome do modelo. O CrewAI agora prefere receber o modelo como string para evitar erros de validação.*

In [ ]:
import os
import pandas as pd
from crewai import Agent, Task, Crew, Process

# INSTRUÇÃO: Substitua 'COLE_SUA_API_KEY_AQUI' pela sua chave do Google AI Studio
API_KEY = "COLE_SUA_API_KEY_AQUI"
os.environ["GOOGLE_API_KEY"] = API_KEY

# CONFIGURAÇÃO TÉCNICA: 
# Passamos o modelo como string no formato "google/modelo" para compatibilidade total com o Pydantic do CrewAI.
llm_config = "google/gemini-2.5-flash"

print(f"✅ Configurado para usar: {llm_config}")

### BLOCO 3: Inteligência de Dados (Carregamento)
*Vamos carregar o dataset de evasão gerado nos labs anteriores.*

In [ ]:
try:
    # Carregando o dataset real da disciplina
    df = pd.read_excel('Lab 05 - EvasaoClientesClassificados.xlsx')
    
    # Enviamos os 5 primeiros clientes para análise cirúrgica (Amostra VIP)
    amostra_clientes = df.head(5).to_markdown() 
    print("✅ Inteligência de dados carregada. Squad pronta para o briefing.")
except:
    print("❌ ERRO: Suba o arquivo 'Lab 05 - EvasaoClientesClassificados.xlsx' no menu lateral do Colab!")

### BLOCO 4: Convocando a Squad (Definição de Agentes)
*Criamos 3 especialistas. Note que passamos 'llm_config' (string) para o parâmetro llm.*

In [ ]:
investigador = Agent(
    role='Analista de Dados (The Investigator)',
    goal='Identificar padrões de risco nos 5 clientes fornecidos e priorizar o mais crítico.',
    backstory='Você é um detetive de dados experiente. Sua especialidade é encontrar o motivo exato pelo qual um cliente cancela.',
    allow_delegation=False,
    llm=llm_config
)

arquiteto = Agent(
    role='Estrategista de Retenção (The Architect)',
    goal='Desenhar uma oferta personalizada de retenção baseada no Nível de Cobertura do cliente.',
    backstory='Você é um mestre em negociação comercial. Sua missão é criar ofertas que o cliente não possa recusar.',
    allow_delegation=False,
    llm=llm_config
)

comunicador = Agent(
    role='Redator Executivo (The Communicator)',
    goal='Escrever um e-mail profissional e acolhedor em Português do Brasil para o cliente.',
    backstory='Você é um especialista em comunicação corporativa. Você sabe transformar dados frios em empatia e valor.',
    allow_delegation=False,
    llm=llm_config
)
print("✅ Squad convocada: Investigador, Arquiteto e Comunicador postos.")

### BLOCO 5: Plano de Ação (Definição de Tarefas)
*Damos ordens específicas para cada agente. O output de um será o input do outro.*

In [ ]:
tarefa_analise = Task(
    description=f"Analise estes clientes e identifique quem tem maior risco de sair: {amostra_clientes}",
    expected_output="Um resumo técnico do cliente mais crítico e o motivo detalhado do risco.",
    agent=investigador
)

tarefa_estrategia = Task(
    description="Com base na análise do Investigador, crie uma proposta de upgrade ou desconto imbatível.",
    expected_output="Uma estratégia de retenção detalhada para o cliente identificado como crítico.",
    agent=arquiteto
)

tarefa_email = Task(
    description="Redija o e-mail final de retenção. O tom deve ser de 'convite premium' e acolhedor.",
    expected_output="O texto completo do e-mail em Português do Brasil pronto para envio (Markdown).",
    agent=comunicador
)
print("✅ Plano de ação traçado. Iniciando orquestração...")

### BLOCO 6: A Operação (Execução da Crew)
*Ativamos a linha de produção. Acompanhe o 'Thought' (pensamento) da IA no log abaixo.*

In [ ]:
squad_retencao = Crew(
    agents=[investigador, arquiteto, comunicador],
    tasks=[tarefa_analise, tarefa_estrategia, tarefa_email],
    process=Process.sequential, # Linha de produção: um após o outro
    verbose=True
)

print("\n🚀 SQUAD EM AÇÃO! Aguarde a conclusão da operação...\n")
resultado_final = squad_retencao.kickoff()

print("\n" + "="*80)
print("📄 RELATÓRIO FINAL DA OPERAÇÃO CHURN ZERO")
print("="*80 + "\n")
print(resultado_final)